In [2]:
pip install pandas openpyxl 

Defaulting to user installation because normal site-packages is not writeable
You should consider upgrading via the '/Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.


In [3]:
import pandas as pd
import numpy as np

In [4]:
import pandas as pd

files = {
    2024: "/Users/valeriia/Desktop/Ala-Tash DA/Строительные материалы 2024.xlsm",
    2025: "/Users/valeriia/Desktop/Ala-Tash DA/Строительные материалы 2025.xlsm",
    2026: "/Users/valeriia/Desktop/Ala-Tash DA/Строительные материалы 2026.xlsm"
}

items_list = []

for year, file_path in files.items():

    df = pd.read_excel(
        file_path,
        sheet_name="Статистика",
        engine="openpyxl"
    )

    # Очищаем названия столбцов
    df.columns = (
        df.columns
        .astype(str)
        .str.replace("\n", " ", regex=False)
        .str.replace("\r", " ", regex=False)
        .str.replace(r"\s+", " ", regex=True)
        .str.strip()
    )

    # Оставляем строки с номером заказа
    df = df[df["№ заказа"].notna()].copy()

    # Номер заказа как текст
    df["№ заказа"] = (
        df["№ заказа"]
        .astype(str)
        .str.strip()
    )

    # Уникальный ID заказа
    df["Год"] = year
    df["Order_ID"] = (
        df["Год"].astype(str)
        + "_"
        + df["№ заказа"]
    )

    # До 10 позиций в одном заказе
    for position in range(10):

        suffix = "" if position == 0 else f".{position}"

        name_col = f"Наименование{suffix}"

        unit_col = (
            "Ед. измер."
            if position == 0
            else f"Ед. измер..{position}"
        )

        qty_col = (
            "Кол-во"
            if position == 0
            else f"Кол-во.{position}"
        )

        price_col = (
            "Цена, сом"
            if position == 0
            else f"Цена, сом.{position}"
        )

        discounted_price_col = (
            "Цена со скидкой, сом"
            if position == 0
            else f"Цена со скидкой, сом.{position}"
        )

        # Первая товарная сумма начинается с .1
        sum_col = f"Сумма, сом.{position + 1}"

        discounted_sum_col = (
            "Сумма со скидкой, сом"
            if position == 0
            else f"Сумма со скидкой, сом.{position}"
        )

        cols = [
            name_col,
            unit_col,
            qty_col,
            price_col,
            discounted_price_col,
            sum_col,
            discounted_sum_col
        ]

        # Если блок существует
        if not all(col in df.columns for col in cols):
            print(f"{year}: блок {position + 1} пропущен")
            continue

        item = df[
            ["Order_ID", "Год", "№ заказа"] + cols
        ].copy()

        item.columns = [
            "Order_ID",
            "Год",
            "№ заказа",
            "Наименование",
            "Ед. измер.",
            "Кол-во",
            "Цена, сом",
            "Цена со скидкой, сом",
            "Сумма позиции, сом",
            "Сумма позиции со скидкой, сом"
        ]

        item["Позиция"] = position + 1

        # Оставляем только реальные позиции
        item = item[
            item["Наименование"].notna()
        ].copy()

        items_list.append(item)

sales_items = pd.concat(
    items_list,
    ignore_index=True
)

In [5]:
numeric_columns = [
    "Кол-во",
    "Цена, сом",
    "Цена со скидкой, сом",
    "Сумма позиции, сом",
    "Сумма позиции со скидкой, сом"
]

for col in numeric_columns:
    sales_items[col] = pd.to_numeric(
        sales_items[col],
        errors="coerce"
    )

In [6]:
sales_items["Наименование"] = (
    sales_items["Наименование"]
    .astype(str)
    .str.strip()
)

sales_items["Ед. измер."] = (
    sales_items["Ед. измер."]
    .astype(str)
    .str.strip()
)

In [7]:
sales_items["Ед. измер."] = (
    sales_items["Ед. измер."]
    .where(sales_items["Ед. измер."].notna())
    .astype("string")
    .str.strip()
)

In [9]:
print(sales_items.shape)
sales_items.head(20)
sales_items["Наименование"].value_counts().head(30)
service_keywords = [
    "фп",
    "фмп",
    "фаска",
    "калибровка",
    "доставка",
    "распил"
]

pattern = "|".join(service_keywords)

sales_items["Тип позиции"] = (
    sales_items["Наименование"]
    .str.lower()
    .str.contains(pattern, na=False)
    .map({
        True: "Услуга",
        False: "Материал"
    })
)
sales_items.to_excel(
    "/Users/valeriia/Desktop/Ala-Tash DA/sales_items_master_v1.xlsx",
    index=False
)

(8512, 12)


In [10]:
print(sales_items.shape)
sales_items.head(10)

(8512, 12)


,Order_ID,Год,№ заказа,Наименование,Ед. измер.,Кол-во,"Цена, сом","Цена со скидкой, сом","Сумма позиции, сом","Сумма позиции со скидкой, сом",Позиция,Тип позиции
0,"2024_0,1",2024,"0,1",Белая луна,кв.м.,1.6900,9500.0,NaN,16055.00,0.0,1,Материал
1,"2024_0,2",2024,"0,2",Габбро,кв.м.,0.6060,12700.0,NaN,7696.20,0.0,1,Материал
2,"2024_0,3",2024,"0,3",Серая сталь,кв.м.,6.7300,10000.0,NaN,67300.00,0.0,1,Материал
3,"2024_0,4",2024,"0,4",Серая волна,кв.м.,5.5600,10600.0,NaN,58936.00,0.0,1,Материал
4,"2024_0,5",2024,"0,5",Серая сталь,кв.м.,3.4360,10000.0,NaN,34360.00,0.0,1,Материал
5,"2024_0,6",2024,"0,6",зеленый обож,кв.м.,0.9600,5750.0,NaN,5520.00,0.0,1,Материал
6,"2024_0,8",2024,"0,8",Эдельвейс,кв.м.,1.9220,5100.0,NaN,9802.20,0.0,1,Материал
7,"2024_0,9",2024,"0,9",Королевский,кв.м.,1.7495,9700.0,NaN,16970.15,0.0,1,Материал
8,"2024_0,10",2024,"0,10",габбро,кв.м.,2.4805,12700.0,NaN,31502.35,0.0,1,Материал
9,"2024_0,11",2024,"0,11",Лабрадорит коричневый,кв.м.,0.6750,10400.0,NaN,7020.00,0.0,1,Материал


In [12]:
import pandas as pd

orders = pd.read_excel(
    "/Users/valeriia/Desktop/Ala-Tash DA/sales_orders_master_v1.xlsx"
)

items = pd.read_excel(
    "/Users/valeriia/Desktop/Ala-Tash DA/sales_items_master_v1.xlsx"
)

In [15]:
items_without_order = items[
    ~items["Order_ID"].isin(orders["Order_ID"])
]

print(items_without_order.shape)
orders["Order_ID"].duplicated().sum()
items["Order_ID"].duplicated().sum()
items["Наименование"].isna().sum()

(0, 12)


np.int64(0)

In [16]:
print(orders.shape)
print(items.shape)

print(orders["Order_ID"].nunique())
print(items["Order_ID"].nunique())

(3758, 28)
(8512, 12)
3749
3226


In [17]:
orders[
    orders["Order_ID"].duplicated(keep=False)
].sort_values("Order_ID")

,№ заказа,Анкета,Ф.И.О.,Установщик,Принят,Оговорено,Отпущен,"Сумма, сом","Скидка, сом",Со скидкой,...,Месяц,Квартал,Статус оплаты,Есть скидка,Категория чека,Дней выполнения,Источник клиента,Тип оплаты,Месяц_Год,Дата_месяца
83,"0,86",NaN,Ширин,Николай,2024-02-01,2024-02-13,2024-02-09,26027.50,0.0,26027.50,...,Февраль,1.0,Переплата,Нет,10–30 тыс.,8.0,Не указан,Обычная продажа,2024-02,2024-02-01
85,"0,86",NaN,Нигай,Тронин А.,2024-02-01,2024-02-08,2024-02-09,6888.60,389.0,6499.60,...,Февраль,1.0,Переплата,Да,до 10 тыс.,8.0,Не указан,Обычная продажа,2024-02,2024-02-01
1340,"13,42",NaN,Гулжан,NaN,2024-10-02,2024-10-11,2024-10-14,46097.00,0.0,46097.00,...,Октябрь,4.0,Переплата,Нет,30–50 тыс.,12.0,Не указан,Обычная продажа,2024-10,2024-10-01
1341,"13,42",NaN,Аладжиани Сергей,NaN,2024-10-03,2024-10-10,2024-10-11,25107.00,7.0,25100.00,...,Октябрь,4.0,Оплачен,Да,10–30 тыс.,8.0,Не указан,Обычная продажа,2024-10,2024-10-01
1349,13.5,отмена,NaN,NaN,NaT,NaT,NaT,NaN,NaN,NaN,...,NaN,NaN,Нет данных,Нет,NaN,NaN,Не указан,Обычная продажа,NaN,NaT
1304,13.5,NaN,Сергеев Андрей,NaN,2024-10-01,2024-10-04,2024-10-07,280300.00,0.0,280300.00,...,Октябрь,4.0,Оплачен,Нет,100–500 тыс.,6.0,Не указан,Обычная продажа,2024-10,2024-10-01
1524,"15,25",NaN,Ширин,Будалов Андрей,2024-11-15,2024-11-26,2024-11-29,63444.00,44.0,63400.00,...,Ноябрь,4.0,Оплачен,Да,50–100 тыс.,14.0,Не указан,Обычная продажа,2024-11,2024-11-01
1523,"15,25",NaN,Асанбаев Канат,Тронин А.,2024-11-15,2024-11-28,2024-11-28,137090.50,0.0,137090.50,...,Ноябрь,4.0,Есть долг,Нет,100–500 тыс.,13.0,Не указан,Обычная продажа,2024-11,2024-11-01
420,"4,21",отмена,NaN,NaN,NaT,NaT,NaT,0.00,0.0,0.00,...,NaN,NaN,Оплачен,Нет,NaN,NaN,Не указан,Обычная продажа,NaN,NaT
421,"4,21",NaN,Камалов Бинали,NaN,2024-04-16,2024-04-24,2024-04-24,3700.00,0.0,3700.00,...,Апрель,2.0,Оплачен,Нет,до 10 тыс.,8.0,Не указан,Обычная продажа,2024-04,2024-04-01


In [19]:
orders[
    orders["№ заказа"] == "0,86"
][[
    "№ заказа",
    "Ф.И.О.",
    "Принят",
    "Сумма, сом",
    "Со скидкой"
]]

,№ заказа,Ф.И.О.,Принят,"Сумма, сом",Со скидкой
83,"0,86",Ширин,2024-02-01,26027.5,26027.5
85,"0,86",Нигай,2024-02-01,6888.6,6499.6
1771,"0,86",NaN,NaT,NaN,NaN
3169,"0,86",Курмамбетов Улан,2026-02-03,52502.0,51579.0


In [18]:
orders_without_items = orders[
    ~orders["Order_ID"].isin(items["Order_ID"])
]

print(orders_without_items.shape)
orders_without_items.head(20)

(525, 28)


,№ заказа,Анкета,Ф.И.О.,Установщик,Принят,Оговорено,Отпущен,"Сумма, сом","Скидка, сом",Со скидкой,...,Месяц,Квартал,Статус оплаты,Есть скидка,Категория чека,Дней выполнения,Источник клиента,Тип оплаты,Месяц_Год,Дата_месяца
6,"0,7",отмена,NaN,NaN,NaT,NaT,NaT,0.0,0.0,0.0,...,NaN,NaN,Оплачен,Нет,NaN,NaN,Не указан,Обычная продажа,NaN,NaT
20,0.21,отмена,NaN,NaN,NaT,NaT,NaT,NaN,NaN,NaN,...,NaN,NaN,Нет данных,Нет,NaN,NaN,Не указан,Обычная продажа,NaN,NaT
22,0.23,отмена,NaN,NaN,NaT,NaT,NaT,NaN,NaN,NaN,...,NaN,NaN,Нет данных,Нет,NaN,NaN,Не указан,Обычная продажа,NaN,NaT
27,"0,28",отмена,отмена,NaN,NaT,NaT,NaT,0.0,0.0,0.0,...,NaN,NaN,Оплачен,Нет,NaN,NaN,Не указан,Обычная продажа,NaN,NaT
31,0.32,отмена,NaN,NaN,NaT,NaT,NaT,NaN,NaN,NaN,...,NaN,NaN,Нет данных,Нет,NaN,NaN,Не указан,Обычная продажа,NaN,NaT
32,0.33,отмена,NaN,NaN,NaT,NaT,NaT,NaN,NaN,NaN,...,NaN,NaN,Нет данных,Нет,NaN,NaN,Не указан,Обычная продажа,NaN,NaT
45,0.46,отмена,NaN,NaN,NaT,NaT,NaT,NaN,NaN,NaN,...,NaN,NaN,Нет данных,Нет,NaN,NaN,Не указан,Обычная продажа,NaN,NaT
56,0.57,отмена,NaN,NaN,NaT,NaT,NaT,NaN,NaN,NaN,...,NaN,NaN,Нет данных,Нет,NaN,NaN,Не указан,Обычная продажа,NaN,NaT
68,0.69,отмена,NaN,NaN,NaT,NaT,NaT,NaN,NaN,NaN,...,NaN,NaN,Нет данных,Нет,NaN,NaN,Не указан,Обычная продажа,NaN,NaT
69,0.7,отмена,NaN,NaN,NaT,NaT,NaT,NaN,NaN,NaN,...,NaN,NaN,Нет данных,Нет,NaN,NaN,Не указан,Обычная продажа,NaN,NaT


In [20]:
orders[
    orders["№ заказа"] == "0,86"
][[
    "№ заказа",
    "Ф.И.О.",
    "Принят",
    "Сумма, сом",
    "Со скидкой"
]]

,№ заказа,Ф.И.О.,Принят,"Сумма, сом",Со скидкой
83,"0,86",Ширин,2024-02-01,26027.5,26027.5
85,"0,86",Нигай,2024-02-01,6888.6,6499.6
1771,"0,86",NaN,NaT,NaN,NaN
3169,"0,86",Курмамбетов Улан,2026-02-03,52502.0,51579.0
